In [31]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
import keras
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

In [32]:
!pip install scikeras

In [33]:
data = pd.read_csv("Churn_Modelling.csv")

data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geography = OneHotEncoder(sparse_output=False)
geography_encoded = onehot_encoder_geography.fit_transform(data[['Geography']])
geography_encoded_df = pd.DataFrame(geography_encoded, columns=onehot_encoder_geography.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography', axis = 1), geography_encoded_df], axis=1)


In [34]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [35]:
X = data.drop('Exited', axis=1)
y = data['Exited']

In [36]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [37]:
import pickle

with open('label_encoder_gender.pkl', 'wb') as f:
    pickle.dump(label_encoder_gender, f)

with open('onehot_encoder_geography.pkl', 'wb') as f:
    pickle.dump(onehot_encoder_geography, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [38]:
def create_model(neurons = 32, layers = 1):
    model = Sequential()

    model.add(Dense(neurons, activation="relu", input_shape = (X_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons, activation="relu"))

    model.add(Dense(1,activation="sigmoid"))

    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=['accuracy'])

    return model


In [39]:
model = KerasClassifier(
    model=create_model,
    layers=1,
    neurons=32,
    epochs=50,
    batch_size=10,
    verbose=0
)

In [42]:
params_grid = {
    "neurons": [32, 64],
    "layers": [1, 2],
    "epochs": [20]

}

In [ ]:
grid = GridSearchCV(estimator=model, param_grid=params_grid,n_jobs=-1,cv = 3)
grid_result = grid.fit(X_train_scaled,y_train)

In [ ]:
print("Best : %f using %s" % (grid_result.best_score_, grid_result.best_params_))